The Look: E-commerce 
----------------------------------------------------------------
Preliminary analysis conducted on 20k sample for computational efficiency.

Dataset from bigquery-public-data: "thelook_ecommerce"
Objective: exploratory data analysis (EDA) to understand data structure, quality, and potential business insights.
Tools: BigQuery + Pandas


In [1]:
#1. IMPORT LIBRARIES & SETUP
  
from dotenv import load_dotenv    #to load .env file
import os                         #interact with Operating System
from google.cloud import bigquery #connect to BigQuery

import pandas as pd
import numpy as np

In [2]:
# 2. CONNECT TO BIGQUERY
load_dotenv()   #load file .env with google cloud project id

PROJECT_ID = os.getenv('PROJECT_ID')
print(f"Project ID: {PROJECT_ID if PROJECT_ID else 'NONE FOUND (create .env file with PROJECT_ID)'}")

client = bigquery.Client(project=PROJECT_ID)
print("Connected to BigQuery successfully!")

Project ID: sql-sandbox-472206
Connected to BigQuery successfully!


In [3]:
# 3. DATA SELECTION  (7 datasets)
queries = {
    "products": """
        SELECT * FROM `bigquery-public-data.thelook_ecommerce.products` 
        LIMIT 20000
    """,
    "users": """
        SELECT * EXCEPT(first_name, last_name, user_geom) 
        FROM `bigquery-public-data.thelook_ecommerce.users` 
        WHERE created_at <= CURRENT_TIMESTAMP()
        LIMIT 20000
    """,
    "orders_raw": """
        SELECT * FROM `bigquery-public-data.thelook_ecommerce.orders`
        WHERE created_at<= CURRENT_TIMESTAMP() AND shipped_at <= CURRENT_TIMESTAMP()
    """,
    "order_items": """
        SELECT * FROM `bigquery-public-data.thelook_ecommerce.order_items`
        WHERE created_at <= CURRENT_TIMESTAMP() AND shipped_at <= CURRENT_TIMESTAMP()
        LIMIT 20000
    """,
    "events": """
        SELECT * EXCEPT (ip_address, postal_code, uri)
        FROM `bigquery-public-data.thelook_ecommerce.events`
        WHERE created_at <= CURRENT_TIMESTAMP()
        LIMIT 100000
    """,
    "inventory_items": """
        SELECT * FROM `bigquery-public-data.thelook_ecommerce.inventory_items`
        WHERE created_at <= CURRENT_TIMESTAMP() AND sold_at <= CURRENT_TIMESTAMP()
        LIMIT 20000
    """,
    "distribution_centers": """
        SELECT * FROM `bigquery-public-data.thelook_ecommerce.distribution_centers`
    """}


In [4]:
# 4. LOAD PRODUCTS 
try:
    products = client.query(queries["products"]).to_dataframe()
    print(products.head(3))
    
except Exception:
    print("Something went wrong.")

      id     cost     category  \
0  13842  2.51875  Accessories   
1  13928  2.33835  Accessories   
2  14115  4.87956  Accessories   

                                                name brand  retail_price  \
0   Low Profile Dyed Cotton Twill Cap - Navy W39S55D    MG          6.25   
1  Low Profile Dyed Cotton Twill Cap - Putty W39S55D    MG          5.95   
2       Enzyme Regular Solid Army Caps-Black W35S45D    MG         10.99   

  department                               sku  distribution_center_id  
0      Women  EBD58B8A3F1D72F4206201DA62FB1204                       1  
1      Women  2EAC42424D12436BDD6A5B8A88480CC3                       1  
2      Women  EE364229B2791D1EF9355708EFF0BA34                       1  


In [5]:
# 4.1 PRODUCTS EDA 
print("PRODUCTS EXPLORATION")
print(products.info())
print("\n", products[["cost","retail_price"]].describe())

PRODUCTS EXPLORATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      20000 non-null  Int64  
 1   cost                    20000 non-null  float64
 2   category                20000 non-null  object 
 3   name                    19998 non-null  object 
 4   brand                   19976 non-null  object 
 5   retail_price            20000 non-null  float64
 6   department              20000 non-null  object 
 7   sku                     20000 non-null  object 
 8   distribution_center_id  20000 non-null  Int64  
dtypes: Int64(2), float64(2), object(5)
memory usage: 1.4+ MB
None

                cost  retail_price
count  20000.000000  20000.000000
mean      28.519070     59.799238
std       30.492835     66.912015
min        0.008300      0.020000
25%       11.970157     25.000000
50%       20.04

In [6]:
print(f"\nTOP 10 Categories by Sum Retail Price:\n{(products.groupby('category')['retail_price'].agg(['mean', 'sum']).sort_values('sum', ascending=False).head(10))}")
print(f"\nWORST 10 Categories by Sum Retail Price:\n{(products.groupby('category')['retail_price'].agg(['mean', 'sum']).sort_values('sum', ascending=True).head(10))}")


TOP 10 Categories by Sum Retail Price:
                                     mean            sum
category                                                
Outerwear & Coats              151.556756  157922.139793
Jeans                           80.487072  105277.090262
Sweaters                        76.059350   88913.379993
Swim                            58.373059   75768.230050
Suits & Sport Coats            138.426539   68797.990015
Fashion Hoodies & Sweatshirts   56.599267   63334.580150
Sleep & Lounge                  48.797122   62557.910108
Dresses                         98.777984   60254.570129
Active                          51.039084   57418.970033
Intimates                       35.165904   57179.760116

WORST 10 Categories by Sum Retail Price:
                           mean           sum
category                                     
Clothing Sets         85.350526   1621.659988
Jumpsuits & Rompers   76.494167   4589.650009
Socks & Hosiery       17.064022   9504.659988
Legg

In [7]:
# 5. LOAD USERS
try:
    users = client.query(queries["users"]).to_dataframe()
    print(users.head())
    
except Exception:
    print("Something went wrong.")

      id                          email  age gender state  \
0  73531        monicamiles@example.com   30      F  Acre   
1  16326    catherinemcneil@example.org   53      F  Acre   
2  54510         jackiewise@example.org   66      F  Acre   
3  87627         lisabishop@example.net   56      F  Acre   
4   1506  christopherherman@example.net   19      M  Acre   

                street_address postal_code  city country  latitude  longitude  \
0        9366 Petersen Streets   69980-000  null  Brasil -8.065346 -72.870949   
1            680 Amber Prairie   69980-000  null  Brasil -8.065346 -72.870949   
2             835 Reyes Bypass   69980-000  null  Brasil -8.065346 -72.870949   
3  099 Kyle Crossing Suite 455   69980-000  null  Brasil -8.065346 -72.870949   
4            252 Gonzalez Neck   69980-000  null  Brasil -8.065346 -72.870949   

  traffic_source                created_at  
0          Email 2020-06-11 09:55:00+00:00  
1         Search 2021-09-23 03:01:00+00:00  
2         S

In [8]:
# 5.1 USERS EDA
print("USERS EXPLORATION")
print(users.info())

USERS EXPLORATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   id              20000 non-null  Int64              
 1   email           20000 non-null  object             
 2   age             20000 non-null  Int64              
 3   gender          20000 non-null  object             
 4   state           20000 non-null  object             
 5   street_address  20000 non-null  object             
 6   postal_code     20000 non-null  object             
 7   city            20000 non-null  object             
 8   country         20000 non-null  object             
 9   latitude        20000 non-null  float64            
 10  longitude       20000 non-null  float64            
 11  traffic_source  20000 non-null  object             
 12  created_at      20000 non-null  datetime64[us, UTC]
dtypes: Int64(2), 

In [9]:
print("Age users \n", users["age"].describe())

Age users 
 count      20000.0
mean       41.0424
std      17.074363
min           12.0
25%           26.0
50%           41.0
75%           56.0
max           70.0
Name: age, dtype: Float64


In [10]:
print("\n", users[['id','email','city']].nunique())
email_duplicate = users[['email']].duplicated(keep=False)
email_duplicated = users.loc[email_duplicate,["id","age", "created_at", "email", "city", "country"]].sort_values('email')
print("\n",email_duplicated)  #We have different users with the same email but age discrepancies and location.


 id       20000
email    19057
city      2035
dtype: int64

           id  age                       created_at  \
11363  32430   12        2024-02-15 06:25:00+00:00   
15327  98926   41        2019-12-02 11:56:00+00:00   
7613   74952   42        2023-09-24 17:19:00+00:00   
4936   15023   30        2019-01-29 04:55:00+00:00   
2075   99578   53        2022-12-13 10:15:00+00:00   
...      ...  ...                              ...   
18024  67178   51 2026-03-13 03:48:14.237909+00:00   
17562  54406   12        2025-03-23 17:44:00+00:00   
3516   77929   21        2023-11-18 12:31:00+00:00   
6920   31122   21        2022-12-24 08:27:00+00:00   
12865  37580   36        2023-04-05 18:59:00+00:00   

                             email                  city        country  
11363       aarongreen@example.net             Los Banos  United States  
15327       aarongreen@example.net               Itapajé         Brasil  
7613         aaronking@example.net                Urumqi          C

In [11]:
print("\n", users['gender'].value_counts(normalize=True))
print("\n", users['country'].value_counts())
print("\n Account creation date:", users['created_at'].agg(['min', 'max']))


 gender
M    0.5039
F    0.4961
Name: proportion, dtype: float64

 country
United States    5469
Brasil           3727
China            3418
Spain            3349
Germany          1406
France           1113
South Korea      1016
Japan             246
Belgium           137
Poland             64
Australia          48
Colombia            7
Name: count, dtype: int64

 Account creation date: min          2019-01-02 07:53:00+00:00
max   2026-03-17 19:18:31.635714+00:00
Name: created_at, dtype: datetime64[ns, UTC]


In [12]:
# 6. LOAD ORDERS 
try:
    orders_raw = client.query(queries["orders_raw"]).to_dataframe()
    print(orders_raw.head())
    
except Exception:
    print("Something went wrong.")

   order_id  user_id    status gender                created_at returned_at  \
0         2        3  Complete      F 2025-03-26 18:52:46+00:00         NaT   
1        13       12  Complete      F 2024-02-23 07:16:14+00:00         NaT   
2        33       32  Complete      F 2021-05-10 14:02:32+00:00         NaT   
3        61       52  Complete      F 2025-09-28 16:45:42+00:00         NaT   
4        63       54  Complete      F 2025-04-12 15:03:16+00:00         NaT   

                 shipped_at              delivered_at  num_of_item  
0 2025-03-27 10:33:46+00:00 2025-03-30 13:46:46+00:00            2  
1 2024-02-26 01:26:14+00:00 2024-02-26 23:53:14+00:00            1  
2 2021-05-10 21:23:32+00:00 2021-05-15 18:53:32+00:00            1  
3 2025-09-30 13:20:42+00:00 2025-10-03 05:23:42+00:00            2  
4 2025-04-13 11:40:16+00:00 2025-04-13 12:57:16+00:00            1  


In [13]:
# 6.1 ORDERS EDA
print("ORDERS EXPLORATION")
print(orders_raw.info())
print("\n", orders_raw["num_of_item"].describe())

ORDERS EXPLORATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80541 entries, 0 to 80540
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   order_id      80541 non-null  Int64              
 1   user_id       80541 non-null  Int64              
 2   status        80541 non-null  object             
 3   gender        80541 non-null  object             
 4   created_at    80541 non-null  datetime64[us, UTC]
 5   returned_at   12453 non-null  datetime64[us, UTC]
 6   shipped_at    80541 non-null  datetime64[us, UTC]
 7   delivered_at  43402 non-null  datetime64[us, UTC]
 8   num_of_item   80541 non-null  Int64              
dtypes: Int64(3), datetime64[us, UTC](4), object(2)
memory usage: 5.8+ MB
None

 count     80541.0
mean     1.452912
std      0.807516
min           1.0
25%           1.0
50%           1.0
75%           2.0
max           4.0
Name: num_of_item, dtype: Float64


In [14]:
print("\n", orders_raw['gender'].value_counts(normalize=True))
print("\n", orders_raw['status'].value_counts(normalize=True))
print("\nRange", orders_raw[['created_at','shipped_at','delivered_at']].agg(['min', 'max']))


 gender
M    0.503731
F    0.496269
Name: proportion, dtype: float64

 status
Shipped     0.461119
Complete    0.384264
Returned    0.154617
Name: proportion, dtype: float64

Range                    created_at                       shipped_at  \
min 2019-01-16 05:10:38+00:00        2019-01-18 01:36:38+00:00   
max 2026-03-18 00:31:59+00:00 2026-03-18 21:30:21.009850+00:00   

                        delivered_at  
min        2019-01-21 16:02:27+00:00  
max 2026-03-23 19:43:04.949127+00:00  


In [15]:
orders_raw['shipping_duration'] = orders_raw['delivered_at'] - orders_raw['shipped_at']
shipping_duration = orders_raw['shipping_duration'].dropna()
print("Orders delivered (%):", round(orders_raw['shipping_duration'].notna().mean() * 100, 2))
print("\nShipping duration (just delivered orders):\n", shipping_duration.describe())


Orders delivered (%): 53.89

Shipping duration (just delivered orders):
 count                     43402
mean     2 days 12:07:01.844154
std      1 days 10:38:03.408995
min             0 days 00:00:00
25%             1 days 06:02:00
50%             2 days 11:54:00
75%             3 days 18:10:00
max             4 days 23:59:00
Name: shipping_duration, dtype: object


In [16]:
in_transit = (orders_raw['shipped_at'].notnull() & orders_raw['delivered_at'].isnull())
transit_orders = orders_raw.loc[in_transit,['num_of_item', 'status', 'created_at', 'shipped_at','delivered_at']].sort_values('shipped_at') 

print("Shipped orders but not yet delivered:\n",transit_orders.agg({'shipped_at': ['count','min','max']}))
print("\n Items in transit (as % of total items):", round(transit_orders['num_of_item'].sum()/(orders_raw['num_of_item'].sum())*100,2))


Shipped orders but not yet delivered:
                              shipped_at
count                             37139
min           2019-02-01 00:20:47+00:00
max    2026-03-18 21:30:21.009850+00:00

 Items in transit (as % of total items): 46.09


In [17]:
# 7. LOAD ORDERS-ITEMS
try:
    order_items = client.query(queries["order_items"]).to_dataframe()
    print(order_items.head())
    
except Exception:
    print("Something went wrong.")

       id  order_id  user_id  product_id  inventory_item_id    status  \
0  152840    105312    84228       14235             411989  Complete   
1   51980     35752    28619       14235             140075   Shipped   
2  127725     87930    70372       14235             344235   Shipped   
3   94256     64882    51909       14159             254079  Complete   
4  104754     72081    57739       14159             282334  Returned   

                 created_at                shipped_at  \
0 2026-01-29 20:29:58+00:00 2026-01-29 20:53:50+00:00   
1 2022-08-12 22:06:18+00:00 2022-08-14 00:15:10+00:00   
2 2021-03-10 21:46:27+00:00 2021-03-11 14:18:50+00:00   
3 2024-12-19 10:48:05+00:00 2024-12-22 03:42:57+00:00   
4 2025-06-26 19:13:38+00:00 2025-06-27 00:47:04+00:00   

               delivered_at               returned_at  sale_price  
0 2026-02-03 19:48:50+00:00                       NaT        0.02  
1                       NaT                       NaT        0.02  
2             

In [18]:
# 7.1 ORDERS-ITEMS EDA
print("ORDER-ITEMS EXPLORATION")
print(order_items.info())
print("\n", order_items['sale_price'].describe())

ORDER-ITEMS EXPLORATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype              
---  ------             --------------  -----              
 0   id                 20000 non-null  Int64              
 1   order_id           20000 non-null  Int64              
 2   user_id            20000 non-null  Int64              
 3   product_id         20000 non-null  Int64              
 4   inventory_item_id  20000 non-null  Int64              
 5   status             20000 non-null  object             
 6   created_at         20000 non-null  datetime64[us, UTC]
 7   shipped_at         20000 non-null  datetime64[us, UTC]
 8   delivered_at       10839 non-null  datetime64[us, UTC]
 9   returned_at        3115 non-null   datetime64[us, UTC]
 10  sale_price         20000 non-null  float64            
dtypes: Int64(5), datetime64[us, UTC](4), float64(1), object(1)
memory usage: 1.8+ MB
N

In [19]:
print(order_items.loc[order_items['sale_price']<0.5]) # to check if there is any discrepancy in the selling prices and the products


       id  order_id  user_id  product_id  inventory_item_id    status  \
0  152840    105312    84228       14235             411989  Complete   
1   51980     35752    28619       14235             140075   Shipped   
2  127725     87930    70372       14235             344235   Shipped   
3   94256     64882    51909       14159             254079  Complete   
4  104754     72081    57739       14159             282334  Returned   
5  137369     94623    75706       14159             370324  Returned   

                 created_at                shipped_at  \
0 2026-01-29 20:29:58+00:00 2026-01-29 20:53:50+00:00   
1 2022-08-12 22:06:18+00:00 2022-08-14 00:15:10+00:00   
2 2021-03-10 21:46:27+00:00 2021-03-11 14:18:50+00:00   
3 2024-12-19 10:48:05+00:00 2024-12-22 03:42:57+00:00   
4 2025-06-26 19:13:38+00:00 2025-06-27 00:47:04+00:00   
5 2025-03-20 13:00:59+00:00 2025-03-22 05:37:48+00:00   

               delivered_at               returned_at  sale_price  
0 2026-02-03 19:48:5

In [20]:
print(products.loc[(products['id'] == 14159) | (products['id'] == 14235)])

         id     cost     category  \
1401  14235  0.00830  Accessories   
2300  14159  0.17738  Accessories   

                                                   name        brand  \
1401         Indestructable Aluminum Aluma Wallet - RED      marshal   
2300  Set of 2 - Replacement Insert For Checkbook Wa...  Made in USA   

      retail_price department                               sku  \
1401          0.02      Women  8425BC94A44E3D1BB3C8C026B2702C00   
2300          0.49      Women  C92B32FBC94E2DFF3E5516401D9BB463   

      distribution_center_id  
1401                       1  
2300                       1  


In [21]:
print("\n", order_items['status'].value_counts(normalize=True))
print("\nRange", order_items[['created_at','shipped_at','delivered_at']].agg(['min', 'max']))


 status
Shipped     0.45805
Complete    0.38620
Returned    0.15575
Name: proportion, dtype: float64

Range                    created_at                       shipped_at  \
min 2019-01-16 03:37:33+00:00        2019-01-18 01:36:38+00:00   
max 2026-03-18 21:23:07+00:00 2026-03-18 21:30:21.009850+00:00   

                        delivered_at  
min        2019-01-22 20:16:38+00:00  
max 2026-03-23 16:42:19.520744+00:00  


In [22]:
# 8. LOAD INVENTORY-ITEMS
try:
    inventory_items = client.query(queries["inventory_items"]).to_dataframe()
    print(inventory_items.sample(3))
    
except Exception:
    print("Something went wrong.")

           id  product_id                created_at                   sold_at  \
19999  268618        7707 2025-06-09 23:19:39+00:00 2025-07-07 10:47:39+00:00   
640    483665       29041 2020-02-25 18:42:08+00:00 2020-03-04 01:48:08+00:00   
8443   164163       14035 2023-05-26 09:29:38+00:00 2023-07-02 18:59:38+00:00   

            cost   product_category  \
19999  19.288361  Blazers & Jackets   
640    15.010000        Accessories   
8443   16.795801        Accessories   

                                            product_name        product_brand  \
19999  BLACK KIMONO DUSTER JACKET LONG STRETCH JERSEY...         LOTUSTRADERS   
640                        Armani Exchange Logo Web Belt  A:X Armani Exchange   
8443                     Suncloud Optics Star Sunglasses             Suncloud   

       product_retail_price product_department  \
19999             52.990002              Women   
640               38.000000                Men   
8443              39.990002              Wo

In [23]:
# 8.1 INVENTORY-ITEMS EDA
print("INVENTORY-ITEMS EXPLORATION")
print(inventory_items.info())

INVENTORY-ITEMS EXPLORATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype              
---  ------                          --------------  -----              
 0   id                              20000 non-null  Int64              
 1   product_id                      20000 non-null  Int64              
 2   created_at                      20000 non-null  datetime64[us, UTC]
 3   sold_at                         20000 non-null  datetime64[us, UTC]
 4   cost                            20000 non-null  float64            
 5   product_category                20000 non-null  object             
 6   product_name                    20000 non-null  object             
 7   product_brand                   20000 non-null  object             
 8   product_retail_price            20000 non-null  float64            
 9   product_department              20000 non-null  object 

In [24]:
print(inventory_items['product_distribution_center_id'].value_counts(normalize=True))
print("\n",inventory_items['product_category'].value_counts(normalize=True))
print("\n",inventory_items['product_brand'].value_counts(normalize=True).head(3))
print("\nData range", inventory_items[['created_at','sold_at']].agg(['min', 'max']))

product_distribution_center_id
2      0.1837
1     0.15795
3     0.11305
9      0.0984
4     0.09335
6      0.0905
7      0.0796
8     0.07905
10    0.06185
5     0.04255
Name: proportion, dtype: Float64

 product_category
Accessories          0.49675
Active               0.44100
Blazers & Jackets    0.06225
Name: proportion, dtype: float64

 product_brand
Allegra K    0.04430
Champion     0.03505
Ray-Ban      0.03190
Name: proportion, dtype: float64

Data range                           created_at                   sold_at
min        2018-11-17 12:16:54+00:00 2019-01-11 08:49:54+00:00
max 2026-03-17 00:46:34.650112+00:00 2026-03-18 21:23:07+00:00


In [25]:
inventory_items['margin'] = inventory_items['product_retail_price'] - inventory_items['cost']
inventory_items['margin_pct'] = inventory_items['margin'] / inventory_items['product_retail_price']
print(inventory_items[['cost','product_retail_price','margin','margin_pct']].describe())   #all products have more than 50% margin

               cost  product_retail_price        margin    margin_pct
count  20000.000000          20000.000000  20000.000000  20000.000000
mean      18.663686             45.715792     27.052106      0.591982
std       25.555134             62.079104     36.689516      0.031165
min        0.008300              0.020000      0.011700      0.530000
25%        6.520500             15.990000      9.546030      0.567000
50%       11.250000             26.990000     16.025000      0.592000
75%       21.710850             52.990002     30.900000      0.617000
max      403.641002            903.000000    532.769998      0.669000


In [26]:
print(inventory_items.groupby('product_department')['margin_pct'].mean().sort_values(ascending=False))

product_department
Women    0.594171
Men      0.590100
Name: margin_pct, dtype: float64


In [27]:
inventory_items['days_to_sell'] = (inventory_items['sold_at'] - inventory_items['created_at']).dt.days
print(inventory_items['days_to_sell'].describe())

count    20000.000000
mean        29.343250
std         17.357819
min          0.000000
25%         14.000000
50%         29.000000
75%         45.000000
max         59.000000
Name: days_to_sell, dtype: float64


In [28]:
print(inventory_items.groupby('product_distribution_center_id')[['days_to_sell','margin_pct']].mean())

                                days_to_sell  margin_pct
product_distribution_center_id                          
1                                  29.462488    0.589264
2                                  29.272727    0.591815
3                                  29.225564    0.587200
4                                  29.171398    0.588650
5                                  27.230317    0.597792
6                                  29.821547    0.592732
7                                  30.152638    0.590155
8                                  28.912081    0.594407
9                                  29.660569    0.601149
10                                 29.481002    0.592760


In [29]:
# 9. LOAD EVENTS
try:
    events = client.query(queries["events"]).to_dataframe()
    print(events.head())
    
except Exception:
    print("Something went wrong.")

        id  user_id  sequence_number                            session_id  \
0  2397022     <NA>                3  048686e1-66ef-44f1-9cbc-7b0f6909ad33   
1  2010611     <NA>                3  9f5d68f1-4cda-4396-9983-7fa5e66959fd   
2  2375357     <NA>                3  1db05660-d5c4-4b94-bab6-422567da2f0b   
3  1564799     <NA>                3  08c3e2de-b00f-4ba3-b29f-2aad485a9d9f   
4  1443451     <NA>                3  dc8ea3f6-7f35-47e4-8110-4a8270b36279   

                 created_at       city      state browser traffic_source  \
0 2024-09-11 14:35:00+00:00    Sapporo   Hokkaido  Chrome       Facebook   
1 2019-02-02 16:42:00+00:00    Sapporo   Hokkaido  Chrome        YouTube   
2 2024-07-10 11:39:00+00:00  São Paulo  São Paulo      IE       Facebook   
3 2019-12-04 18:33:00+00:00  São Paulo  São Paulo  Chrome          Email   
4 2019-01-27 01:15:00+00:00  São Paulo  São Paulo  Chrome          Email   

  event_type  
0     cancel  
1     cancel  
2     cancel  
3     cancel  

In [30]:
# 9.1 EVENTS EDA
print("EVENTS EXPLORATION")
print(events.info())

EVENTS EXPLORATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype              
---  ------           --------------   -----              
 0   id               100000 non-null  Int64              
 1   user_id          23146 non-null   Int64              
 2   sequence_number  100000 non-null  Int64              
 3   session_id       100000 non-null  object             
 4   created_at       100000 non-null  datetime64[us, UTC]
 5   city             100000 non-null  object             
 6   state            100000 non-null  object             
 7   browser          100000 non-null  object             
 8   traffic_source   100000 non-null  object             
 9   event_type       100000 non-null  object             
dtypes: Int64(3), datetime64[us, UTC](1), object(6)
memory usage: 7.9+ MB
None


In [31]:
print(events['state'].unique())   #For info about location user, better to merge with users table. 

['Hokkaido' 'São Paulo' 'Aomori' 'Grand Est' 'Beijing' 'New York'
 'Extremadura' 'Inner Mongolia Autonomous Region' 'Brussels' 'Berlin'
 'Remote territory' 'Tokyo' 'Liaoning' 'Andalucía' 'Occitanie' 'Seoul'
 'Auvergne-Rhône-Alpes' 'Comunidad Valenciana' 'Massachusetts' 'Sachsen'
 'Brandenburg' 'Jilin' "Provence-Alpes-Côte d'Azur" 'Castilla-La Mancha'
 'Wallonia' 'Normandie' 'Heilongjiang' 'Galicia' 'Pennsylvania' 'Flanders'
 'Cataluña' 'Mecklenburg-Vorpommern' 'Nouvelle-Aquitaine'
 'Centre-Val de Loire' 'Delaware' 'Gangwon-do' 'Hauts-de-France'
 'Shanghai' 'District of Columbia' 'País Vasco' 'Virginia'
 'New South Wales' 'Corse' 'Hamburg' 'Rio de Janeiro' 'Maryland'
 'Kanagawa' 'Bourgogne-Franche-Comté' 'Jiangsu' 'Niedersachsen'
 'Schleswig-Holstein' 'Bretagne' 'Anhui' 'Shandong' 'La Rioja'
 'Australian Capital Territory' 'Chiba' 'West Virginia' 'North Carolina'
 'Comunidad de Madrid' 'Bremen' 'Rhode Island' 'South Carolina'
 'Espírito Santo' 'Sachsen-Anhalt' 'Daejeon' 'Victoria' 'Shan

In [32]:
print(events.agg({'user_id': ['count','size','nunique'], 'session_id': ['count','size','nunique']}))
print("\n NaN user_id %:",events['user_id'].isna().mean() * 100)

         user_id  session_id
count      23146      100000
size      100000      100000
nunique    14044       93248

 NaN user_id %: 76.854


In [33]:
logged_users = events['user_id'].notna()
print(pd.crosstab( events['event_type'],logged_users, normalize='index')) #all cancel is from non logged users (NAN)
print("\n",pd.crosstab(events['traffic_source'],logged_users, normalize='index'))

user_id        False     True 
event_type                    
cancel      1.000000  0.000000
cart        0.416816  0.583184

 user_id            False     True 
traffic_source                    
Adwords         0.770383  0.229617
Email           0.766983  0.233017
Facebook        0.768403  0.231597
Organic         0.770821  0.229179
YouTube         0.769099  0.230901


In [34]:
# 10. LOAD DISTRIBUTION CENTERS
try:
    distribution_centers = client.query(queries["distribution_centers"]).to_dataframe()
    print(distribution_centers.info())
    
except Exception:
    print("Something went wrong.")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id                        10 non-null     Int64  
 1   name                      10 non-null     object 
 2   latitude                  10 non-null     float64
 3   longitude                 10 non-null     float64
 4   distribution_center_geom  10 non-null     object 
dtypes: Int64(1), float64(2), object(2)
memory usage: 542.0+ bytes
None


In [35]:
# 10.1 DISTRIBUTION CENTERS EDA
print(distribution_centers.head(10).sort_values(by='id'))

   id                                         name  latitude  longitude  \
5   1                                   Memphis TN   35.1174   -89.9711   
9   2                                   Chicago IL   41.8369   -87.6847   
0   3                                   Houston TX   29.7604   -95.3698   
7   4                               Los Angeles CA   34.0500  -118.2500   
3   5                               New Orleans LA   29.9500   -90.0667   
6   6  Port Authority of New York/New Jersey NY/NJ   40.6340   -73.7834   
1   7                              Philadelphia PA   39.9500   -75.1667   
8   8                                    Mobile AL   30.6944   -88.0431   
2   9                                Charleston SC   32.7833   -79.9333   
4  10                                  Savannah GA   32.0167   -81.1167   

  distribution_center_geom  
5  POINT(-89.9711 35.1174)  
9  POINT(-87.6847 41.8369)  
0  POINT(-95.3698 29.7604)  
7     POINT(-118.25 34.05)  
3    POINT(-90.0667 29.95)  
